# 22 · Told what the client reads: the slice QP, the loop filter, I or P

Notebook 18's told model gets the slice QP. The client reads two more things off every frame (desktop-streamer,
`stream-h264.js`): whether the loop filter runs (NVENC: yes, the host's x264 line: no) and whether it is an I frame.
The same slice QP means a sharper, deblocked picture from NVENC and a blockier one from x264, so a model told only the
number has to blend the two. Here notebook 7's body is trained told all three, at two seeds, with the shared recipe
(`training.fit`), and set beside the blind model (three seeds) and notebook 18's model told the QP alone (two seeds).

All of them ship the same way: the conditions are a straight line into the biases, `b + W c` (notebook 19), so the
client computes 72 numbers a frame from what it read, and the graph is the blind one's.

Scores on notebook 0's validation split, by line and frame type, then by line and slice QP bin on P frames.

Needs: `00_data_preprocess.ipynb`, and the scores of notebooks 1, 4 and 8 (the blind model) and 18 (told the QP).

In [1]:
"""Settings."""

import os

import numpy as np
import torch

os.environ["DATASETS_DIR"] = "/mnt/d/datasets"   # where the datasets are: set before helpers is imported

from helpers import models, patches, training
from helpers.training import DATA_DIR, DEVICE

NOTEBOOK = 22
WIDTH, BLOCKS, STEM = 24, 2, 3                # notebook 7's body
TOLD = ("qp", "deblocked", "intra")           # slice QP / 51, the loop filter, I frame
COLUMNS = patches.columns(TOLD)
NAME = "told QP, loop filter, I frame"
SEEDS = (0, 1)
RUN_DIR = training.CHECKPOINT_DIR / "qp_runs"

manifest = training.load_manifest()
SCALE = manifest["scale"]
recorded = training.recorded()
bicubic = recorded[1, "bicubic"]
blind = {name: s for (notebook, name), s in recorded.items()
         if notebook in (4, 8) and name.startswith("luma 24 + 2 blocks")}
told_qp = {name: s for (notebook, name), s in recorded.items() if notebook == 18 and name.startswith("told the QP")}


def build(seed=0):
    torch.manual_seed(seed)
    return models.ToldLuma(len(COLUMNS), WIDTH, BLOCKS, STEM, scale=SCALE).to(DEVICE)


def run_file(seed):
    return RUN_DIR / f"told_qp_frame_seed{seed}.pt"

## 1. Train

`training.fit` at seeds 0 and 1, each run saved to `checkpoints/qp_runs/`.

In [ ]:
runs = {}
for seed in SEEDS:
    model = build(seed)
    history, best = training.fit(model, run_file(seed), manifest, told=TOLD, seed=seed, verbose=0)
    runs[seed] = model.eval()
    print(f"seed {seed}: best epoch {best['epoch']:3} of {len(history):3}, select SSIM-Y {best['score']:.4f}")

## 2. Gain over bicubic

Every model on the validation split; then by line and frame type, and by line and slice QP bin on P frames. Each kind
of model is the mean of its seeds, and the last column is how far apart the two seeds of this notebook's model land.

In [ ]:
scores = {}
for seed, model in runs.items():
    name = NAME + (f", seed {seed}" if seed else "")
    scores[name] = training.val_scores(lambda lr, condition, model=model: model(lr, condition[:, COLUMNS]), manifest)
    training.record(NOTEBOOK, name, scores[name])
training.show(training.compare({"bicubic": bicubic, **blind, **told_qp, **scores}, manifest, base=bicubic))

files = manifest["splits"]["val"]
labels = {name: np.load(DATA_DIR / files[name]) for name in ("qp", "encoder", "intra")}
kinds = {"blind": list(blind.values()), "told the QP": list(told_qp.values()), NAME: list(scores.values())}
gain = {kind: {key: np.mean([s[key] - bicubic[key] for s in runs_], axis=0) for key in ("ssim", "y")}
        for kind, runs_ in kinds.items()}


def row(label, inside):
    if not inside.any():
        return
    cells = [f"{gain[kind]['ssim'][inside].mean():+.4f} / {gain[kind]['y'][inside].mean():+.2f}" for kind in kinds]
    gap = abs(scores[NAME]["ssim"][inside].mean() - scores[f"{NAME}, seed 1"]["ssim"][inside].mean())
    print(f"{label:<18}{inside.sum():8}{bicubic['ssim'][inside].mean():9.4f}" + "".join(f"{c:>34}" for c in cells)
          + f"{gap:10.4f}")


header = f"{'':<18}{'patches':>8}{'bicubic':>9}" + "".join(f"{kind + ', SSIM-Y / dB':>34}" for kind in kinds) + f"{'seed gap':>10}"
print("\n" + header)
row("all", np.ones(len(labels["qp"]), bool))
for e, encoder in enumerate(manifest["encoders"]):
    for intra, frame in ((False, "P"), (True, "I")):
        row(f"{encoder} {frame}", (labels["encoder"] == e) & (labels["intra"] == intra))
print("\n" + header)
edges = manifest["qp_edges"]
for e, encoder in enumerate(manifest["encoders"]):
    for low, high in zip(edges[:-1], edges[1:]):
        row(f"{encoder} P {low}-{high}",
            (labels["encoder"] == e) & ~labels["intra"] & (labels["qp"] >= low) & (labels["qp"] < high))

## 3. What it says

*(written after the run)*